In [38]:
#%pip install python-docx
#%pip install contractions

In [1]:
import numpy as np
import pandas as pd
import sys,os
import csv
import string
import matplotlib.pyplot as plt
from docx import Document

import contractions
import nltk
#nltk.download("punkt")
#nltk.download("punkt_tab")
from nltk.tokenize import word_tokenize
#nltk.download("stopwords")
from nltk.corpus import stopwords
#nltk.download("wordnet")
from nltk.stem import WordNetLemmatizer

from nltk import pos_tag
#nltk.download('averaged_perceptron_tagger_eng')
from collections import Counter

In [11]:
# MAC Family segment values (that are included into the first level model, i.e. > 0.2

# 1     0.000010
# 2     0.000010
# 3     0.274794
# 4     0.000010
# 5     0.000010
# 6     0.282058
# 7     0.321516
# 8     0.000010
# 9     0.000010
# 10    0.000010
# 11    0.206130
# 12    0.000010
# 13    0.000010
# 14    0.000010
# 15    0.000010
# 16    0.000010
# 17    0.000010
# 18    0.320696
# 19    0.000010
# 20    0.000010
# 21    0.213816
# 22    0.000010

In [29]:
# dirs
data_dir = 'data_SepSegments/'

# Define subjects
participants = [49, 58, 95, 115, 127, 181, 186, 190, 191, 200, 201] + list(range(206, 238)) + list(range(239, 254))

# load group number of participants
GroupIndex = pd.read_csv("Narratives participants.csv")

# split
GroupIndex[['GroupNumber', 'TaskOrder']] = GroupIndex['Notes'].str.split(',', n=1, expand=True)

GroupIndex['GroupNumber'] = GroupIndex['GroupNumber'].str.strip()
GroupIndex['TaskOrder'] = GroupIndex['TaskOrder'].str.strip()

In [23]:
group2_ids = GroupIndex.loc[GroupIndex["GroupNumber"] == "group2", "BIDS ID"].tolist()
group2_ids


['sub-049',
 'sub-058',
 'sub-186',
 'sub-190',
 'sub-221',
 'sub-222',
 'sub-223',
 'sub-224',
 'sub-225',
 'sub-226',
 'sub-227',
 'sub-228',
 'sub-229',
 'sub-230',
 'sub-231',
 'sub-232',
 'sub-233',
 'sub-234',
 'sub-235',
 'sub-236',
 'sub-237']

In [40]:
processedTextData = pd.DataFrame(columns=['participant','word_count_noStop_noFiller', 'sentence_count',
                                          'CC','CD','DT','EX','FW','IN','JJ','JJR','JJS','LS',
                                          'MD','NN','NNS','NNP','NNPS','PDT','POS','PRP','PRP$','RB',
                                          'RBR','RBS','RP','SYM','TO','UH','VB','VBD','VBG','VBN',
                                          'VBP','VBZ','WDT','WP','WP$','WRB',
                                          'nouns','verbs','adjectives','adverbs','pronouns',
                                          'conjunctions','prepositions','interjections','articles',
                                          'nouns_to_verbs_ratio','content_words_prop','function_words_prop','pronouns_to_nouns_ratio',
                                          'temporal_connectives_count','causal_connectives_count','GroupNumber','TaskOrder'])

# existing tags? https://www.ling.upenn.edu/courses/Fall_2003/ling001/penn_treebank_pos.html
# 1.	CC	Coordinating conjunction
# 2.	CD	Cardinal number
# 3.	DT	Determiner
# 4.	EX	Existential there
# 5.	FW	Foreign word
# 6.	IN	Preposition or subordinating conjunction
# 7.	JJ	Adjective
# 8.	JJR	Adjective, comparative
# 9.	JJS	Adjective, superlative
# 10.	LS	List item marker
# 11.	MD	Modal
# 12.	NN	Noun, singular or mass
# 13.	NNS	Noun, plural
# 14.	NNP	Proper noun, singular
# 15.	NNPS	Proper noun, plural
# 16.	PDT	Predeterminer
# 17.	POS	Possessive ending
# 18.	PRP	Personal pronoun
# 19.	PRP$	Possessive pronoun
# 20.	RB	Adverb
# 21.	RBR	Adverb, comparative
# 22.	RBS	Adverb, superlative
# 23.	RP	Particle
# 24.	SYM	Symbol
# 25.	TO	to
# 26.	UH	Interjection
# 27.	VB	Verb, base form
# 28.	VBD	Verb, past tense
# 29.	VBG	Verb, gerund or present participle
# 30.	VBN	Verb, past participle
# 31.	VBP	Verb, non-3rd person singular present
# 32.	VBZ	Verb, 3rd person singular present
# 33.	WDT	Wh-determiner
# 34.	WP	Wh-pronoun
# 35.	WP$	Possessive wh-pronoun
# 36.	WRB	Wh-adverb

# Define connectives and fillers
TEMPORAL_CONNECTIVES = ['then', 'after', 'after that', 'first', 'next', 'finally', 'meanwhile']
CAUSAL_CONNECTIVES = ['because', 'so', 'therefore', 'as a result', 'thus']


for participant in participants:

    print('processing participant: ',participant)
    
    if participant == 219:
        print('no data')
    elif participant == 220:
        print('no data')
    elif participant == 242:
        print('no data')
    else:

        bids_id = f"sub-{participant:03d}"   # format e.g., 49 → sub-049
        filename = os.path.join(data_dir, f"{bids_id}_time.xlsx")

        # Check if participant is in group2
        if bids_id in group2_ids:
            if os.path.exists(filename):
                df = pd.read_excel(filename)
                print(f"✅ Loaded {filename} with shape {df.shape}")
        
                df = df[['Unnamed: 0','Transcript']]
            
                # Select rows of high MAC Family (note that xlsx files of recall start with event 1 and not event 0
                selected = df[df["Unnamed: 0"].isin([3, 6, 7, 11, 18, 21])]["Transcript"]
                
                # Combine into one continuous text
                full_text = " ".join(selected.dropna().astype(str))
        
                # count number of sentences
                sentences = nltk.sent_tokenize(full_text)
            
                text = full_text.lower()
            
                # Expand contractions 
                text = contractions.fix(text)
            
                # tokenize
                tokens = word_tokenize(text)
        
                # extract amount of nouns, verbs etc.
                # POS tagging
                pos_tags = pos_tag(tokens)
            
                # Count POS
                pos_counts = Counter(tag for word, tag in pos_tags)
    
                # Initialize row dictionary with zeros for all columns
                row = {col: 0 for col in processedTextData.columns}
            
                # Update POS counts in the row dictionary
                for tag, count in pos_counts.items():
                    if tag in row:  # only update if the tag exists
                        row[tag] = count
        
                # Map POS tags to broader categories
                categories = {
                    'nouns': ['NN', 'NNS', 'NNP', 'NNPS'],
                    'verbs': ['VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ'],
                    'adjectives': ['JJ', 'JJR', 'JJS'],
                    'adverbs': ['RB', 'RBR', 'RBS'],
                    'pronouns': ['PRP', 'PRP$', 'WP', 'WP$'],
                    'conjunctions': ['CC'],
                    'prepositions': ['IN'],
                    'interjections': ['UH'],
                    'articles': ['DT']
                }
                
                # Compute category counts
                cat_counts = {cat: sum(pos_counts[tag] for tag in tags if tag in pos_counts) 
                              for cat, tags in categories.items()}
                
                # Compute proportions
                cat_props = {cat: count / len(tokens) if len(tokens) > 0 else 0 
                         for cat, count in cat_counts.items()}
            
                # Lexical diversity
                lexical_diversity = len(set(tokens)) / len(tokens) if len(tokens) > 0 else 0
        
                # add all categories proportions to the dataframe
                row['nouns'] = cat_props['nouns']
                row['verbs'] = cat_props['verbs']
                row['adjectives'] = cat_props['adjectives']
                row['adverbs'] = cat_props['adverbs']
                row['pronouns'] = cat_props['pronouns']
                row['conjunctions'] = cat_props['conjunctions']
                row['prepositions'] = cat_props['prepositions']
                row['interjections'] = cat_props['interjections']
                row['articles'] = cat_props['articles']
            
                # Narrative metrics
                row['nouns_to_verbs_ratio'] = cat_props['nouns']/cat_props['verbs'] if cat_props['verbs']>0 else 0
                row['content_words_prop'] = cat_props['nouns'] + cat_props['verbs'] + cat_props['adjectives'] + cat_props['adverbs']
                row['function_words_prop'] = cat_props['pronouns'] + cat_props['conjunctions'] + cat_props['prepositions'] + cat_props['articles']
                row['pronouns_to_nouns_ratio'] = cat_props['pronouns']/cat_props['nouns'] if cat_props['nouns']>0 else 0
            
                # Connective counts
                row['temporal_connectives_count'] = sum(tokens.count(c) for c in TEMPORAL_CONNECTIVES)
                row['causal_connectives_count'] = sum(tokens.count(c) for c in CAUSAL_CONNECTIVES)
                
                ## to count total number of words without including stopwords and filler words
                # Remove punctuation & stopwords
                stop_words = set(stopwords.words("english"))
                tokens = [t for t in tokens if t not in string.punctuation and t not in stop_words]
            
                filler_words = {"um", "uh", "erm", "mm", "ah", "like", "you know"}
                tokens = [t for t in tokens if t not in filler_words]
            
                # Lemmatize
                lemmatizer = WordNetLemmatizer()
                tokens = [lemmatizer.lemmatize(t) for t in tokens]
        
                row['word_count_noStop_noFiller'] = len(tokens)
                row['sentence_count'] = len(sentences)
        
                # add group info
                bids_id = "sub-%03d" % participant
        
                groupNumber = GroupIndex.loc[GroupIndex['BIDS ID'] == bids_id, 'GroupNumber'].values[0]
                taskOrder = GroupIndex.loc[GroupIndex['BIDS ID'] == bids_id, 'TaskOrder'].values[0]
        
                row['GroupNumber'] = groupNumber
                row['TaskOrder'] = taskOrder
        
                row['participant'] = participant
                
                # Append row to DataFrame
                # Convert row dict to DataFrame
                row_df = pd.DataFrame([row])
                
                # Concatenate to main DataFrame
                processedTextData = pd.concat([processedTextData, row_df], ignore_index=True)

            else:
                print(f"⚠️ File not found: {filename}")
        else:
            print(f"⏭️ Skipping {bids_id}, not in group2")
    
# Save DataFrame to CSV
processedTextData.to_csv('processedTextData_highMACFamily.csv', index=False)
            

processing participant:  49
✅ Loaded ./data_SepSegments/sub-049_time.xlsx with shape (24, 8)
processing participant:  58
✅ Loaded ./data_SepSegments/sub-058_time.xlsx with shape (25, 8)
processing participant:  95
⏭️ Skipping sub-095, not in group2
processing participant:  115
⏭️ Skipping sub-115, not in group2
processing participant:  127
⏭️ Skipping sub-127, not in group2
processing participant:  181
⏭️ Skipping sub-181, not in group2
processing participant:  186
✅ Loaded ./data_SepSegments/sub-186_time.xlsx with shape (25, 8)
processing participant:  190
✅ Loaded ./data_SepSegments/sub-190_time.xlsx with shape (24, 8)
processing participant:  191
⏭️ Skipping sub-191, not in group2
processing participant:  200
⏭️ Skipping sub-200, not in group2
processing participant:  201
⏭️ Skipping sub-201, not in group2
processing participant:  206
⏭️ Skipping sub-206, not in group2
processing participant:  207
⏭️ Skipping sub-207, not in group2
processing participant:  208
⏭️ Skipping sub-208, n

/var/folders/2b/y7vqbp2s3slchmzyrsz69p1m0000gn/T/ipykernel_1426/283433858.py:185: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  processedTextData = pd.concat([processedTextData, row_df], ignore_index=True)
